In [26]:
!pip install shimmy>=2.0
!pip install stable-baselines3 gym numpy pandas

import gym
from gym import spaces
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

In [29]:
class StockDataIterator:
    def __init__(self, file_path, lookback_window, chunk_size):
        self.data = pd.read_csv(file_path)
        self.lookback_window = lookback_window
        self.chunk_size = chunk_size
        self.current_chunk = 0

    def get_data(self, step, lookback_window):
        end_idx = step + lookback_window
        data_chunk = self.data.iloc[step:end_idx, :]
        # Select features for the model (like Open, High, Low, Close, Volume)
        return data_chunk[['open', 'high', 'low', 'close', 'volume']].values

# StockTradingEnv to simulate stock trading
class StockTradingEnv(gym.Env):
    def __init__(self, data_iterator):
        super(StockTradingEnv, self).__init__()
        self.data_iterator = data_iterator
        self.current_step = 0
        self.action_space = spaces.Discrete(3)  # Buy, Sell, Hold
        self.observation_space = spaces.Box(low=0, high=1, shape=(data_iterator.lookback_window, 5), dtype=np.float32)

    def reset(self):
        self.current_step = 0
        return self._get_observation()

    def _get_observation(self):
        data = self.data_iterator.get_data(self.current_step, self.data_iterator.lookback_window)
        return data

    def step(self, action):
        # Placeholder for reward calculation (you can use a more complex reward function)
        data = self.data_iterator.get_data(self.current_step, self.data_iterator.lookback_window)
        reward = 0  # Modify with actual reward logic (e.g., profit/loss)

        # Simple condition to simulate action behavior (buy, sell, hold)
        if action == 0:  # Buy
            reward = np.random.uniform(-1, 1)
        elif action == 1:  # Sell
            reward = np.random.uniform(-1, 1)
        else:  # Hold
            reward = 0

        self.current_step += 1
        done = self.current_step >= len(self.data_iterator.data) - self.data_iterator.lookback_window
        return self._get_observation(), reward, done, {}

    def render(self):
        # Optional: implement rendering logic if  want to visualize the trading process
        pass




In [30]:
# Create the iterator for stock data
data_iterator = StockDataIterator(
    file_path="/content/historical_data.csv",
    lookback_window=10,  # previous days to look at
    chunk_size=10000     # Chunk size for data
)

# Creating and wrap the environment
def make_env():
    return StockTradingEnv(data_iterator=data_iterator)

env = DummyVecEnv([make_env])

# Initializing PPO with appropriate policy kwargs
policy_kwargs = dict(
    net_arch=dict(
        pi=[64, 64],  # Policy network
        vf=[64, 64]   # Value function network
    )
)

model = PPO(
    "MlpPolicy",
    env,
    policy_kwargs=policy_kwargs,
    learning_rate=0.0001,
    n_steps=4096,
    batch_size=128,
    n_epochs=5,
    gamma=0.98,
    gae_lambda=0.9,
    clip_range=0.1,
    verbose=1
)


# Train the model
try:
    model.learn(total_timesteps=10000)
    # Save the model
    model.save("ppo_stock_trading_model")
    print("Training completed successfully!")
except Exception as e:
    print(f"Error during training: {str(e)}")


    #references: openai, github, finrl documentation

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Using cpu device
-----------------------------
| time/              |      |
|    fps             | 469  |
|    iterations      | 1    |
|    time_elapsed    | 8    |
|    total_timesteps | 4096 |
-----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 446           |
|    iterations           | 2             |
|    time_elapsed         | 18            |
|    total_timesteps      | 8192          |
| train/                  |               |
|    approx_kl            | 0.00047963913 |
|    clip_fraction        | 0             |
|    clip_range           | 0.1           |
|    entropy_loss         | -1.1          |
|    explained_variance   | -0.404        |
|    learning_rate        | 0.0001        |
|    loss                 | 0.593         |
|    n_updates            | 5             |
|    policy_gradient_loss | -0.00111      |
|    value_loss           | 1.27          |
-------------------------